In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from urllib.parse import urlparse

# 1. On passe à 50 000 lignes
print("Chargement de 50 000 lignes...")
df = pd.read_csv('../data/archive (2)/new_data_urls.csv').drop_duplicates(subset=['url'])
df = df.sample(n=50000, random_state=42).copy()

# 2. Extraction du domaine
def extract_domain(url):
    try:
        if not isinstance(url, str): return "inconnu"
        if not url.startswith('http'): url = 'http://' + url
        return urlparse(url).netloc
    except: return "inconnu"
df['domaine'] = df['url'].apply(extract_domain)

# 3. Séparation honnête
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['domaine']))
df_train = df.iloc[train_idx]
df_test = df.iloc[test_idx]

# 4. N-grammes (on garde 2000 features pour que ça reste rapide)
print("Création des n-grammes...")
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=2000)
X_train = vectorizer.fit_transform(df_train['url'])
X_test = vectorizer.transform(df_test['url'])
y_train = df_train['status']
y_test = df_test['status']

# 5. Entraînement
print("Entraînement (1 à 2 minutes)...")
model_ngram = RandomForestClassifier(n_estimators=30, random_state=42, n_jobs=-1)
model_ngram.fit(X_train, y_train)

# 6. Évaluation
y_pred = model_ngram.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nPrécision avec 50 000 lignes : {accuracy * 100:.2f} %")
print("\n--- Rapport détaillé ---")
print(classification_report(y_test, y_pred, target_names=["Phishing (0)", "Sain (1)"]))

Chargement de 50 000 lignes...
Création des n-grammes...
Entraînement (1 à 2 minutes)...

Précision avec 50 000 lignes : 92.50 %

--- Rapport détaillé ---
              precision    recall  f1-score   support

Phishing (0)       0.94      0.91      0.92      4804
    Sain (1)       0.91      0.94      0.93      4991

    accuracy                           0.92      9795
   macro avg       0.93      0.92      0.92      9795
weighted avg       0.93      0.92      0.92      9795



In [3]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

print("Entraînement de XGBoost avec les n-grammes...")
start = time.time()

# On utilise tree_method="hist" pour que ça aille vite
model_xgb_ngram = XGBClassifier(
    n_estimators=100, 
    random_state=42, 
    n_jobs=-1, 
    tree_method="hist", 
    eval_metric="logloss"
)

# On utilise les données X_train et X_test qui sont DÉJÀ en mémoire (les n-grammes)
model_xgb_ngram.fit(X_train, y_train)
print(f"Temps d'entraînement : {time.time() - start:.2f} secondes")

# Évaluation
y_pred_xgb = model_xgb_ngram.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"\nPrécision de XGBoost (avec n-grammes) : {accuracy_xgb * 100:.2f} %")
print("\n--- Rapport détaillé ---")
print(classification_report(y_test, y_pred_xgb, target_names=["Phishing (0)", "Sain (1)"]))

Entraînement de XGBoost avec les n-grammes...
Temps d'entraînement : 17.83 secondes

Précision de XGBoost (avec n-grammes) : 93.29 %

--- Rapport détaillé ---
              precision    recall  f1-score   support

Phishing (0)       0.95      0.91      0.93      4804
    Sain (1)       0.92      0.95      0.94      4991

    accuracy                           0.93      9795
   macro avg       0.93      0.93      0.93      9795
weighted avg       0.93      0.93      0.93      9795



In [4]:
import joblib

# On sauvegarde le modèle XGBoost (le cerveau)
joblib.dump(model_xgb_ngram, '../src/model_xgb_ngram.pkl')

# On sauvegarde le vectorizer (le traducteur de texte)
joblib.dump(vectorizer, '../src/vectorizer_ngram.pkl')

print("Modèle XGBoost et Vectorizer sauvegardés avec succès !")

Modèle XGBoost et Vectorizer sauvegardés avec succès !


In [6]:
import shap
import numpy as np
import pandas as pd

# 1. On prépare le "détective" SHAP
explainer = shap.TreeExplainer(model_xgb_ngram)
feature_names = vectorizer.get_feature_names_out()

def expliquer_url(url_texte):
    # On transforme l'URL en n-grammes
    url_vect = vectorizer.transform([url_texte])
    
    # On calcule les valeurs SHAP
    shap_values = explainer.shap_values(url_vect)
    
    # On récupère les valeurs (XGBoost renvoie souvent un array direct)
    if isinstance(shap_values, list):
        vals = shap_values[0][0]
    else:
        vals = shap_values[0]
        
    # On associe chaque n-gramme à son score
    contributions = pd.DataFrame({
        'ngramme': feature_names,
        'impact': vals
    })
    
    # On trie du plus négatif au plus positif
    contributions = contributions.sort_values(by='impact', ascending=True)
    
    print(f"\n--- Analyse de l'URL : {url_texte} ---")
    
    print("\n🚨 Top 5 des raisons qui poussent vers PHISHING (Danger) :")
    # Les impacts négatifs poussent vers le phishing
    for i in range(5):
        print(f"  -> Le texte '{contributions.iloc[i]['ngramme']}' (Impact : {contributions.iloc[i]['impact']:.3f})")
        
    print("\n✅ Top 5 des raisons qui poussent vers SAIN (Sécurisé) :")
    # Les impacts positifs poussent vers le sain
    for i in range(1, 6):
        print(f"  -> Le texte '{contributions.iloc[-i]['ngramme']}' (Impact : +{contributions.iloc[-i]['impact']:.3f})")

# Test
url_suspecte = "http://paypal-login-verify.xyz/secure-update"
expliquer_url(url_suspecte)


--- Analyse de l'URL : http://paypal-login-verify.xyz/secure-update ---

🚨 Top 5 des raisons qui poussent vers PHISHING (Danger) :
  -> Le texte 'pay' (Impact : -1.192)
  -> Le texte 'xyz' (Impact : -0.922)
  -> Le texte 'cure' (Impact : -0.890)
  -> Le texte 'veri' (Impact : -0.633)
  -> Le texte 'verif' (Impact : -0.591)

✅ Top 5 des raisons qui poussent vers SAIN (Sécurisé) :
  -> Le texte 'al-' (Impact : +0.232)
  -> Le texte '.com' (Impact : +0.142)
  -> Le texte 'ttp' (Impact : +0.062)
  -> Le texte '.php' (Impact : +0.039)
  -> Le texte '://' (Impact : +0.035)
